# Trabajo Práctico Integrador - Introducción al Análisis de datos
## Entrega 5: Análisis exploratorio de Datos.

## Caso: análisis y predicción de cancelaciones en reservas hoteleras.

## Lenguaje utilizado: Python y SQL


# Datos de identificación

- **Apellido y nombre:** Rodriguez Joaquin - Pichulman Miguel
- **Comisión:** 12
- **Dataset asignado:** L
- **Entrega:** 5
- **Fecha:** 25 / 10 / 2026

## Planificación del problema predictivo

**1. Objetivo predictivo:**
Construir un modelo de clasificación para predecir si esa reserva será finalmente cancelada (`is_canceled`).

**2. Posibles variables predictoras seleccionadas:**
Priorizamos la información que el hotel ya conoce al momento de concretar la reserva:
- `lead_time`: Tiempo de anticipación de la reserva.
- `hotel`: Tipo de hotel (Urbano o Resort).
- `market_segment`: Segmento de mercado.
- `deposit_type`: Tipo de depósito realizado.
- `customer_type`: Tipo de cliente.
- `previous_cancellations`: Cantidad de cancelaciones anteriores del cliente.

**3. Revisión de variables (Exclusiones para evitar fuga de información):**
- **Identificador:** `booking_id` se excluye porque es solo una etiqueta única para cada fila y no tiene valor predictivo.
- **Fuga de información (*Data Leakage*):** Excluimos `assigned_room_type` y `booking_changes`. Ambas ocurren o pueden modificarse después de que la reserva fue creada, por lo que usarlas equivaldría a darle al modelo información del futuro.

# Preparación del entorno y reconstrucción del modelo

In [4]:
#@title Preparación automática del entorno (ejecutar sin modificar)
import importlib.util
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

print("Preparando el entorno...")

for package in ["duckdb", "pandas", "scikit-learn"]:
    module = "sklearn" if package == "scikit-learn" else package
    if importlib.util.find_spec(module) is None:
        print(f"Instalando {package}...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package
        ])

import duckdb
import pandas as pd

print("Paquetes preparados correctamente.")

DATA_URL = "https://raw.githubusercontent.com/joacoxr1/TPI-hotelbookinggrupoLOriginal/main/hotel%20booking%20TPI%20grupo%20L%20Original.csv"
DATA_PATH = Path("hotel_booking.csv")

if not DATA_PATH.exists():
    print("Descargando dataset...")
    urlretrieve(DATA_URL, DATA_PATH)

print("Cargando dataset...")
df_hotel = pd.read_csv(DATA_PATH)

# Si la celda se ejecuta otra vez, cerramos primero una conexión previa.
if "con" in globals():
    try:
        con.close()
    except Exception:
        pass

con = duckdb.connect(":memory:")
con.register("df_hotel_temp", df_hotel)
con.execute("CREATE OR REPLACE TABLE reservas AS SELECT * FROM df_hotel_temp")

reservas_count = f"{len(df_hotel):,}".replace(",", ".")
print(f"Entorno listo: {reservas_count} reservas y {df_hotel.shape[1]} variables.")

Preparando el entorno...
Paquetes preparados correctamente.
Descargando dataset...
Cargando dataset...
Entorno listo: 25.000 reservas y 32 variables.


## Selección de variable con SQL

In [5]:
consulta = """
SELECT
    is_canceled,
    lead_time,
    market_segment,
    deposit_type,
    customer_type,
    previous_cancellations,
    hotel
FROM reservas
"""
df_modelo = con.sql(consulta).df()
print("Valores faltantes por columna:\n", df_modelo.isnull().sum())
display(df_modelo.head())

Valores faltantes por columna:
 is_canceled               0
lead_time                 0
market_segment            0
deposit_type              0
customer_type             0
previous_cancellations    0
hotel                     0
dtype: int64


,is_canceled,lead_time,market_segment,deposit_type,customer_type,previous_cancellations,hotel
0,1,302,Groups,Non Refund,Transient,0,City Hotel
1,1,65,Groups,Non Refund,Transient,0,City Hotel
2,0,32,Online TA,No Deposit,Transient,0,City Hotel
3,0,3,Online TA,No Deposit,Transient,0,Resort Hotel
4,0,5,Offline TA/TO,No Deposit,Transient,0,Resort Hotel


## Preparación y división de datos

In [12]:

from sklearn.model_selection import train_test_split

y = df_modelo["is_canceled"]
X = df_modelo.drop(columns=["is_canceled"])

# Primero separamos los datos
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Identificamos las variables categóricas
categoricas = [
    "market_segment",
    "deposit_type",
    "customer_type",
    "hotel"
]

# Codificamos usando solo las columnas de entrenamiento
X_train = pd.get_dummies(
    X_train,
    columns=categoricas,
    drop_first=True
)

# Aplicamos las mismas columnas al conjunto de prueba
X_test = pd.get_dummies(
    X_test,
    columns=categoricas,
    drop_first=True
)

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

print("Casos de entrenamiento:", len(X_train))
print("Casos de prueba:", len(X_test))
print("Variables finales:", X_train.shape[1])


Casos de entrenamiento: 17500
Casos de prueba: 7500
Variables finales: 14


## Entrenamiento y predicción

In [7]:
from sklearn.tree import DecisionTreeClassifier

modelo = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
probabilidades = modelo.predict_proba(X_test)[:, 1]

resultados = X_test.copy()
resultados["Real (is_canceled)"] = y_test.values
resultados["Prediccion"] = y_pred
resultados["Probabilidad"] = probabilidades.round(3)
resultados["Acierto"] = resultados["Real (is_canceled)"] == resultados["Prediccion"]

display(resultados.head(12))

,lead_time,previous_cancellations,market_segment_Complementary,market_segment_Corporate,market_segment_Direct,market_segment_Groups,market_segment_Offline TA/TO,market_segment_Online TA,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Group,customer_type_Transient,customer_type_Transient-Party,hotel_Resort Hotel,Real (is_canceled),Prediccion,Probabilidad,Acierto
24052,34,0,False,False,False,False,True,False,False,False,False,False,False,False,1,0,0.214,False
20918,3,0,False,True,False,False,False,False,False,False,False,True,False,True,1,0,0.073,False
20046,2,0,False,True,False,False,False,False,False,False,False,True,False,True,0,0,0.073,True
5581,216,0,False,False,True,False,False,False,False,False,False,True,False,True,1,0,0.214,False
13007,2,3,False,True,False,False,False,False,False,False,False,True,False,False,0,0,0.073,True
18450,62,0,False,False,False,False,False,True,False,False,False,True,False,False,0,0,0.430,True
6475,62,0,False,False,False,False,False,True,False,False,False,True,False,False,0,0,0.430,True
24550,87,0,False,False,False,True,False,False,True,False,False,True,False,False,1,1,1.000,True
12578,15,0,False,False,False,False,True,False,True,False,False,True,False,False,1,1,1.000,True
9440,16,0,False,False,False,False,False,True,False,False,False,True,False,True,0,0,0.430,True


## **Observación de las predicciones:**
En la tabla de resultados podemos ver que el modelo asigna una probabilidad de cancelación a cada reserva y, según si supera o no el umbral (0.5), predice un valor categórico (0 o 1). Observamos casos donde el modelo acierta (la columna `Acierto` es *True*) y otros donde comete errores de clasificación (falsos positivos o falsos negativos).

**Utilidad de estas predicciones:**
Si el hotel logra estimar de antemano el riesgo de que una reserva sea cancelada, podría optimizar sus operaciones. Por ejemplo, permitiría ajustar de manera inteligente las estrategias de *overbooking* (sobreventa) para maximizar la ocupación sin riesgo de dejar pasajeros sin habitación, o diseñar campañas de fidelización para asegurar a los clientes dudosos.

**Limitaciones a considerar antes de decidir:**
1. **Asimetría en los costos de error:** Si el modelo predice erróneamente que un cliente va a cancelar y el hotel decide sobrevender esa habitación (falso positivo), el daño a la reputación y los costos de reubicación pueden ser inmensos si el cliente finalmente asiste.
2. **Contexto estático vs. dinámico:** El árbol aprendió patrones pasados, pero no tiene en cuenta variables externas que cambian constantemente, como crisis económicas repentinas, cambios en las políticas de vuelo, huelgas de transporte o nuevas ofertas de la competencia, lo cual degradaría rápidamente su capacidad predictiva en el mundo real.

## Cierre

In [8]:
if "con" in globals():
    con.close()